# Two agents and a wall

A folder of customer records, and a billing fault nobody in the building knows the rule for. The local model is the only thing that reads a record or writes a fix. The hosted model gets the shape of the problem and sends back a plan.

In [ ]:
from pathlib import Path
from shutil import copytree
from tempfile import mkdtemp

from fastcore.all import L
from ramabana.agent import Agent
from ramabana.tools import LocalHost
from ramabana.vault import VaultHost
from vishalakshi import Vault
from vishalakshi.pii import pii_report

LOCAL, CLOUD, WIRE = 'gemma-e4b', 'sonnet', []

The wall, in four functions. `ingest` builds the vault from a folder into a temp file, so a stateless container needs no volume. `sealed` scans a prompt before the hosted agent can see it. `inside` holds the files and no network; `outside` holds the web and an empty folder.

In [ ]:
class Leak(Exception): pass

def ingest(folder, private=(), public=()):
    v = Vault(Path(mkdtemp())/'vault.db')
    v.add(str(folder))
    for p in private: v.mark_pii(str(p), reason='marked by a person')
    for p in public: v.mark_not_pii(str(p), reason='cleared by a person')
    return v

def sealed(agent, ner=True):
    ask = agent.ask
    def guard(prompt, **kw):
        if (r := pii_report(prompt, ner=ner)).has_pii: raise Leak(dict(r.identifying))
        WIRE.append(prompt)
        return ask(prompt, **kw)
    agent.ask = guard
    return agent

def inside(vault, folder, model=LOCAL):
    return Agent(VaultHost([str(folder)], vault=vault, web=False, index=False), model=model)

def outside(model=CLOUD):
    return sealed(Agent(LocalHost([mkdtemp()], index=False), model=model, readonly=True))

Patterns are arithmetic; judgement is not. A person marks 8851 (a first name and a relationship) and the callback note (case numbers), and clears the sandbox card in the runbook.

In [ ]:
folder = Path(mkdtemp())/'inbox'
copytree('inbox', folder)

v = ingest(folder,
           private=[folder/'letters/refund-8851.md', folder/'notes/callback-2024-03-14.md'],
           public=[folder/'ops/refund-runbook.md'])
L(v.docs).map(lambda d: (Path(d['source']).name, v.pii(d['source'], ner=True).has_pii))

`refuse` holds the answer back. `local` routes the same question to the model on the device, and nothing is sent anywhere.

In [ ]:
q = 'which refunds came back from the bank, and how much is still outstanding?'
v.ask(q, pii='refuse').answer, v.ask(q, pii='local').answer

The brief is a schema rather than prose: shape and quantity, scanned once more before it is allowed out.

In [ ]:
BRIEF = 'problem:str, n_cases:int, rails:str, constraint:str, outcome:str'

def brief(vault, question, schema=BRIEF):
    b = vault.ask(question, pii='local', schema=schema, sections=6).fields
    if pii_report(str(b), ner=True).has_pii: raise Leak(b)
    return b

b = brief(v, 'what is going wrong across these refund cases, in shape and quantity only?')
b

The hosted agent may only propose, and has no path to a record to propose about.

In [ ]:
ASK = """A back office refunds collections taken under direct debit mandates and cannot show you the records.
{problem} Across {n_cases} cases the money went out on {rails}, and {outcome}. {constraint}
Find the scheme rule governing the return of a collected amount, and give me a numbered runbook."""

up = outside()
plan = up.ask(ASK.format(**b))
print(plan)

The local agent applies the plan over the real files.

In [ ]:
APPLY = """Rewrite ops/refund-runbook.md to follow this procedure. Then list every open case it changes, and where each refund should go.

{plan}"""

down = inside(v, folder)
print(down.ask(APPLY.format(plan=plan)))

Everything that crossed the wall, in full.

In [ ]:
print(WIRE[0])
L(WIRE).map(lambda t: pii_report(t, ner=True).has_pii)

Swap the folder, the models and the schema. The four functions are the whole pattern, and the brief is the only thing that crosses.